# Projet : Système de recommandation d'images

## Groupe

- Mathis LAMBERT
- Quentin BOUCHOT
- Hugo RODRIGUES


## Présentation du projet

Comme indiqué dans [Projet.md](fr/Projet/Projet.md), l'objectif global est de construire un **système de recommandation d'images** capable de proposer des images à des utilisateurs selon leurs préférences.

Le projet mobilise plusieurs étapes : collecte des données, enrichissement des images, analyse, visualisation puis recommandation.

Pour notre étude, nous avons choisi de travailler sur un jeu de données de **fleurs**, car il contient des images riches en couleurs et bien adaptées à une première analyse visuelle avec `KMeans`.

Dataset choisi : [pufanyi/flowers102](https://huggingface.co/datasets/pufanyi/flowers102)


## Tâche 1 : Collecte de données

D'après les consignes du projet, cette première tâche demande de collecter au moins **100 images sous licence libre** ainsi que leurs **métadonnées**.

Dans ce notebook, nous utilisons un échantillon de 200 images du dataset Flowers 102 afin de disposer d'une base simple pour tester ensuite l'extraction des couleurs dominantes.


In [ ]:
!pip install datasets

In [ ]:
from datasets import load_dataset

ds = load_dataset("pufanyi/flowers102", split="train", streaming=True)

samples = []
for i, sample in enumerate(ds):
    if i == 200:
        break
    samples.append(sample)

len(samples)

## Préparation des images

Nous convertissons les images en RGB puis nous les stockons dans un `DataFrame` pandas.

Cela permet ensuite d'ajouter facilement de nouvelles colonnes pour les résultats du clustering.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans

df = pd.DataFrame({
    "label": [sample["label"] for sample in samples],
    "image": [sample["image"].convert("RGB") for sample in samples],
})

df.head()

## Clustering des couleurs

Nous allons maintenant définir deux fonctions simples :

- une fonction qui redimensionne l'image ;
- une fonction qui applique `KMeans` sur les pixels RGB.

Chaque pixel est considéré comme un point dans un espace à 3 dimensions : rouge, vert et bleu.


In [ ]:
def resize_image(image, max_size=128):
    image = image.copy()
    image.thumbnail((max_size, max_size))
    return image

def cluster_colors(image, n_colors=5):
    resized_image = resize_image(image)
    
    # Convertir l'image en un tableau de pixels normalisé, on divise par 255 pour avoir des valeurs entre 0 et 1
    pixels = np.array(resized_image, dtype=np.float32) / 255.0
    
    # Reshaper le tableau de pixels pour qu'il soit en 2D, où chaque ligne correspond à un pixel et les colonnes correspondent aux canaux de couleur (R, G, B)
    pixels_2d = pixels.reshape(-1, 3)

    # Appliquer le clustering KMeans aux pixels de l'image
    model = KMeans(n_clusters=n_colors, random_state=42, n_init="auto")
    labels = model.fit_predict(pixels_2d)

    # Obtenir les couleurs dominantes et reconstruire l'image segmentée
    centers = model.cluster_centers_
    segmented_pixels = centers[labels].reshape(pixels.shape)
    segmented_image = (segmented_pixels * 255).astype(np.uint8)

    # Calculer la fréquence de chaque couleur dominante
    counts = np.bincount(labels)
    
    # Trier les couleurs et leurs fréquences par ordre décroissant de fréquence
    order = np.argsort(counts)[::-1]

    # Obtenir la palette de couleurs triée et les fréquences correspondantes
    palette = centers[order]
    frequencies = counts[order]

    return resized_image, segmented_image, palette, frequencies

## Application sur toutes les images

Nous appliquons la même méthode à chaque image de l'échantillon.

On ajoute ensuite quatre colonnes au `DataFrame` :

- l'image redimensionnée ;
- l'image segmentée ;
- la palette de couleurs dominantes ;
- la fréquence de chaque couleur.


In [ ]:
results = [cluster_colors(image, n_colors=5) for image in df["image"]]

df["resized_image"] = [result[0] for result in results]
df["clustered_image"] = [result[1] for result in results]
df["palette"] = [result[2] for result in results]
df["frequencies"] = [result[3] for result in results]

df[["label"]].head()

## Visualisation des résultats

Pour quelques images, nous affichons :

1. l'image originale redimensionnée ;
2. l'image reconstruite après clustering ;
3. la palette des couleurs dominantes.


In [ ]:
def palette_bar(colors, frequencies, width=300, height=50):
    frequencies = frequencies / frequencies.sum()
    bar = np.zeros((height, width, 3), dtype=np.uint8)

    start = 0
    for color, freq in zip(colors, frequencies):
        end = start + int(freq * width)
        bar[:, start:end] = (color * 255).astype(np.uint8)
        start = end

    return bar

sample_ids = [0, 1, 2, 3]
fig, axes = plt.subplots(len(sample_ids), 3, figsize=(12, 3 * len(sample_ids)))

for row, i in enumerate(sample_ids):
    axes[row, 0].imshow(df.loc[i, "resized_image"])
    axes[row, 0].set_title("Image originale")
    axes[row, 0].axis("off")

    axes[row, 1].imshow(df.loc[i, "clustered_image"])
    axes[row, 1].set_title("Image segmentée")
    axes[row, 1].axis("off")

    bar = palette_bar(df.loc[i, "palette"], df.loc[i, "frequencies"])
    axes[row, 2].imshow(bar)
    axes[row, 2].set_title("Palette dominante")
    axes[row, 2].axis("off")

plt.tight_layout()
plt.show()